In [8]:
import cv2
import os
import numpy as np
import pandas as pd 
import skimage
import torch

import byotrack
import byotrack.dataset.ctc as ctc_data


from byotrack.implementation.linker.frame_by_frame.koft import KOFTLinker, KOFTLinkerParameters
from byotrack.implementation.optical_flow.opencv import OpenCVOpticalFlow

from byotrack.implementation.optical_flow.skimage import SkimageOpticalFlow


In [9]:
# Reproduced from CTC submission link.py script

def get_average_size(detections_sequence: list[byotrack.Detections]) -> float:
    """Get the average size of cells in the dataset"""

    total_size = 0
    count = 0
    for detections in detections_sequence:
        if len(detections) <= 0:
            continue

        count += len(detections)
        total_size += int(detections.mass.sum().item())

    return total_size / (count + (count == 0))


def get_average_min_dist(detections_sequence: list[byotrack.Detections]) -> float:
    """Get the average minimal distance between cells in the dataset"""
    sum_min_dist = 0.0
    count = 0
    for detections in detections_sequence:
        if len(detections) <= 1:
            continue

        count += 1
        sum_min_dist += torch.cdist(detections.position, detections.position).sort(dim=1).values[:, 1].median().item()

    return sum_min_dist / (count + (count == 0))


def get_spot_radius_anisotropy(video, detections_sequence):
    anisotropy = 0.0
    
    # Set default parameters
    if video.ndim == 4:  # 2D
        anisotropy = 1.0

    # Anisotropy is computed if not given based on detections (Depends on direciton)
    if video.ndim == 5 and anisotropy <= 0.0:  # 3D
        sizes = sum(
            detections.bbox[:, detections.dim :].to(torch.float32).mean(axis=0) for detections in detections_sequence
        ) / len(detections_sequence)
        anisotropy = float(sizes[1:].mean() / sizes[0])

    # Useful features for setting the parameters
    spot_size = get_average_size(detections_sequence)

    if video.ndim == 5:  # 3D + T + C
        spot_radius = float((spot_size * anisotropy * 3 / 4 / np.pi) ** (1 / 3))  # area = 4/3 pi R^3 / ani
    else:
        spot_radius = float(np.sqrt(spot_size / np.pi))  # pi R^2
    return spot_radius, anisotropy

In [10]:
root_dir = '/home/ddon0001/PhD/data/trackastra_training/draga'

summary_df = pd.read_csv('/home/ddon0001/PhD/experiments/trackastra_training_tracktour/ds_summary.csv')

out_root = '/home/ddon0001/PhD/experiments/trackastra_training_koft/'

In [12]:
errored = ['celegans_dispim_nih_diSPIM_deconv_1',
 'mskcc-confocal_mskcc_confocal_s3_370_isotropic',
 'mskcc-confocal_mskcc_confocal_s1_400_isotropic',
 'mskcc-confocal_mskcc_confocal_s2_376_isotropic']
for row in summary_df.itertuples():
    if row.ds_name not in errored:
        continue
    ds_name = row.ds_name
    im_path = row.im_path
    seg_path = row.seg_path
    out_path = os.path.join(out_root, ds_name, 'RES/')

    if os.path.exists(out_path):
        print(f"Skipping {ds_name} as output path already exists.")
        continue

    # load and transform video as per CTC submission link.py script
    video = byotrack.Video(im_path)
    video.set_transform(
        byotrack.VideoTransformConfig(
            aggregate=True,
            normalize=True,
            compute_stats_on=50 if video.ndim == 4 else 10,
        )
    )

    # extract detections from seg files as per CTC submission link.py script
    try:
        detections_sequence = ctc_data.GroundTruthDetector().run(byotrack.Video(seg_path))
    except TypeError as e:
        print(f"Error processing {ds_name}: {e}")
        errored.append(ds_name)
        continue
    spot_radius, anisotropy = get_spot_radius_anisotropy(video, detections_sequence)
    closest_spot_dist = get_average_min_dist(detections_sequence)
    association_threshold = max(spot_radius * 3, closest_spot_dist)

    # config reproduced from CTC submission link.py script
    if video.ndim == 5:  # 3D, but tvl1 is quite slow and unprecise in CTC
        optflow = SkimageOpticalFlow(
            skimage.registration.optical_flow_tvl1,
            downscale=4,
            parameters={"attachment": 10},
        )
    else:
        # Config reproduced from https://byotrack.readthedocs.io/en/latest/run_examples/Linkers.html#KOFTLinker
        optflow = OpenCVOpticalFlow(cv2.FarnebackOpticalFlow.create(winSize=20), downscale=4)

    specs = KOFTLinkerParameters(
        association_threshold=2e3,  # Most important parameter: don't link if the association likelihood is smaller than 1e-3.
        detection_std=3.0,  # Detections location are precise up to 3.0 pixels
        process_std=1.5,  # Kalman filter predictions are precise up to 3.0 pixels
        flow_std=1.0,  # Optical flow predictions are precise up to 1.0 pixels/frame
        kalman_order=1,  # Order of the kalman filter
        n_gap=5,  # Allow to link after 5 consecutive missed detections
        cost="likelihood",  # See Cost? to see which other cost are available, by default it uses Euclidean distance (And association threshod should be express in pixels)
        split_factor=1 # allow for divisions
    )

    linker = KOFTLinker(specs, optflow)

    tracks = linker.run(video, detections_sequence)
    if len(tracks) == 0:
        print(f"No tracks found for {ds_name}.")
        errored.append(ds_name)
        continue

    try:
        ctc_data.save_tracks(
            out_path,
            tracks,
            detections_sequence=detections_sequence,
            default_radius=spot_radius * 3,
            shape=video.shape[1:],
            n_digit=len(str(len(video)))+1,
            anisotropy=anisotropy,
        )
    except ValueError as e:
        print(f"Error saving tracks for {ds_name}: {e}")
        errored.append(ds_name)
        continue

KOFT linking: 100%|██████████| 270/270 [00:41<00:00,  6.46it/s]


No tracks found for celegans_dispim_nih_diSPIM_deconv_1.


KOFT linking:  32%|███▏      | 69/213 [2:23:31<4:59:31, 124.80s/it]00:02, 58.75it/s]


KeyboardInterrupt: 

In [7]:
errored

['celegans_dispim_nih_diSPIM_deconv_1',
 'mskcc-confocal_mskcc_confocal_s3_370_isotropic',
 'mskcc-confocal_mskcc_confocal_s1_400_isotropic',
 'mskcc-confocal_mskcc_confocal_s2_376_isotropic']

In [13]:
from traccuracy.loaders import load_ctc_data
from traccuracy.matchers import CTCMatcher
from traccuracy.metrics import CTCMetrics
from traccuracy.utils import export_graphs_to_geff

errored = ['celegans_dispim_nih_diSPIM_deconv_1',
 'mskcc-confocal_mskcc_confocal_s3_370_isotropic',
 'mskcc-confocal_mskcc_confocal_s1_400_isotropic',
 'mskcc-confocal_mskcc_confocal_s2_376_isotropic']

for row in summary_df.itertuples():
    if row.ds_name in errored:
        continue
    gt_path = row.tra_gt_path
    res_path = os.path.join(out_root, row.ds_name, 'RES/')
    out_metrics_path = os.path.join(out_root, row.ds_name, 'matched_solution.zarr')
    if os.path.exists(out_metrics_path):
        print(f"Skipping {row.ds_name} as metrics already computed.")
        continue

    print(f"Processing {row.ds_name}...")
    gt_data = load_ctc_data(gt_path)
    res_data = load_ctc_data(res_path)
    matcher = CTCMatcher()
    matched = matcher.compute_mapping(gt_data, res_data)
    metrics = CTCMetrics()
    results = metrics.compute(matched)
    export_graphs_to_geff(out_metrics_path, matched, [results])
        

Processing deepcell_train-80...


Computing node attributes: 100%|██████████| 65/65 [00:00<00:00, 121.60it/s]
1 non-connected masks at t=4.
3 non-connected masks at t=8.
1 non-connected masks at t=9.
2 non-connected masks at t=10.
1 non-connected masks at t=12.
1 non-connected masks at t=14.
3 non-connected masks at t=18.
5 non-connected masks at t=22.
3 non-connected masks at t=23.
3 non-connected masks at t=24.
1 non-connected masks at t=26.
2 non-connected masks at t=27.
2 non-connected masks at t=28.
1 non-connected masks at t=29.
4 non-connected masks at t=30.
2 non-connected masks at t=32.
3 non-connected masks at t=33.
2 non-connected masks at t=34.
2 non-connected masks at t=35.
2 non-connected masks at t=37.
1 non-connected masks at t=38.
1 non-connected masks at t=40.
2 non-connected masks at t=41.
1 non-connected masks at t=42.
1 non-connected masks at t=43.
1 non-connected masks at t=44.
3 non-connected masks at t=46.
2 non-connected masks at t=47.
1 non-connected masks at t=48.
4 non-connected masks at t=4

Processing deepcell_train-25...


Evaluating FN edges: 100%|██████████| 926/926 [00:00<00:00, 376203.56it/s]


Processing deepcell_train-62...


...

Processing vanvliet_recA-151029-11...


Computing node attributes: 100%|██████████| 24/24 [00:00<00:00, 365.28it/s]
1 non-connected masks at t=10.
2 non-connected masks at t=12.
1 non-connected masks at t=15.
1 non-connected masks at t=16.
1 non-connected masks at t=17.
2 non-connected masks at t=18.
Evaluating FN edges: 100%|██████████| 1202/1202 [00:00<00:00, 699632.72it/s]


Processing deepsea_stem-set_21...


Computing node attributes: 100%|██████████| 30/30 [00:00<00:00, 206.12it/s]


ValueError: Segmentation shapes must match between gt and pred